# Triton Tutorial: Writing GPU Kernels in Python

## What is Triton?

**Triton** is a language and compiler by OpenAI for writing GPU kernels in Python. It lets you:

- Write GPU code that's **as fast as hand-tuned CUDA** (often within 90-95%)
- Use **familiar Python syntax** instead of C/C++
- Automatically handle many low-level GPU optimizations (memory coalescing, shared memory, etc.)

### Why use Triton over...

| | PyTorch | Triton | CUDA |
|---|---|---|---|
| **Ease** | Easiest | Medium | Hardest |
| **Control** | Least | Good | Full |
| **Performance** | Good | Great | Best |
| **Use case** | Standard ops | Custom fused ops | Maximum perf |

### Key Concept: Block-based Programming

Unlike CUDA where you think about individual **threads**, Triton operates on **blocks** of data.
Each Triton "program" processes a tile/block of elements at once. The compiler handles
the thread-level details for you.

In [1]:
# First, install triton if needed
# !pip install triton torch

import torch
import triton
import triton.language as tl

print(f"Triton version: {triton.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Triton version: 3.6.0
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5090


---
## 1. Vector Addition — Your First Triton Kernel

The simplest kernel: `output[i] = x[i] + y[i]`

### Anatomy of a Triton kernel

1. **`@triton.jit`** — Decorator that JIT-compiles the function into GPU code
2. **`pid = tl.program_id(0)`** — Each "program instance" gets a unique ID (like blockIdx in CUDA)
3. **`tl.arange(0, BLOCK_SIZE)`** — Create a range of offsets within the block
4. **`tl.load()` / `tl.store()`** — Read from / write to GPU memory
5. **`mask`** — Prevents out-of-bounds memory access

In [2]:
@triton.jit
def add_kernel(
    x_ptr,        # pointer to first input
    y_ptr,        # pointer to second input
    output_ptr,   # pointer to output
    n_elements,   # total number of elements
    BLOCK_SIZE: tl.constexpr,  # compile-time constant
):
    # Each program instance handles one block of BLOCK_SIZE elements.
    # pid tells us WHICH block we are.
    pid = tl.program_id(axis=0)

    # Compute the offsets this program instance will process
    # If BLOCK_SIZE=1024 and pid=2, offsets = [2048, 2049, ..., 3071]
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)

    # Mask to avoid out-of-bounds access on the last block
    mask = offsets < n_elements

    # Load, compute, store
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    output = x + y
    tl.store(output_ptr + offsets, output, mask=mask)

In [4]:
def add(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """Wrapper that launches the Triton kernel."""
    assert x.is_cuda and y.is_cuda
    output = torch.empty_like(x)
    n_elements = output.numel()
    print(n_elements)

    # The "grid" defines how many program instances to launch.
    # We need ceil(n_elements / BLOCK_SIZE) programs.
    # grid is a function of `meta` dict that contains kernel constants.
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)

    # Launch! The [grid] syntax is Triton's way of specifying the launch grid.
    add_kernel[grid](x, y, output, n_elements, BLOCK_SIZE=1024)
    return output


# Test it
torch.manual_seed(0)
size = 98_432  # intentionally not a multiple of 1024 to test masking
x = torch.rand(size, device='cuda')
y = torch.rand(size, device='cuda')

triton_output = add(x, y)
torch_output = x + y

print(f"Max difference: {(triton_output - torch_output).abs().max():.2e}")
print("Correct!" if torch.allclose(triton_output, torch_output) else "MISMATCH!")

98432
Max difference: 0.00e+00
Correct!


### How It Works — Visual Breakdown

```
Input:  [a0, a1, a2, a3, a4, a5, a6, a7, ...]   (n_elements = 98432)
         |--- Block 0 ---|--- Block 1 ---|---...
         pid=0 (1024 el)  pid=1 (1024 el)

Last block (pid=96): only 128 elements → mask protects the rest
```

Key takeaways:
- `tl.constexpr` parameters are known at **compile time** → the compiler can optimize aggressively
- The `mask` pattern is essential for correctness when data isn't perfectly divisible
- `triton.cdiv` = ceiling division helper

---
## 2. Fused Softmax — Why Triton Shines

In PyTorch, `softmax(x)` requires **multiple kernel launches** (max, subtract, exp, sum, divide).
With Triton, we can **fuse** these into a single kernel, eliminating intermediate memory reads/writes.

This is the #1 reason to use Triton: **kernel fusion** for memory-bound operations.

In [5]:
@triton.jit
def softmax_kernel(
    output_ptr,
    input_ptr,
    input_row_stride,   # stride between rows in the input
    output_row_stride,
    n_cols,
    BLOCK_SIZE: tl.constexpr,  # must be >= n_cols
):
    # Each program handles one ROW of the input matrix
    row_idx = tl.program_id(0)

    row_start_ptr = input_ptr + row_idx * input_row_stride
    col_offsets = tl.arange(0, BLOCK_SIZE)
    mask = col_offsets < n_cols

    # Load the entire row into SRAM (on-chip memory)
    row = tl.load(row_start_ptr + col_offsets, mask=mask, other=-float('inf'))

    # Numerically stable softmax: subtract max before exp
    row_max = tl.max(row, axis=0)
    numerator = tl.exp(row - row_max)
    denominator = tl.sum(numerator, axis=0)
    softmax_output = numerator / denominator

    # Store result
    out_row_start = output_ptr + row_idx * output_row_stride
    tl.store(out_row_start + col_offsets, softmax_output, mask=mask)

In [6]:
def softmax(x: torch.Tensor) -> torch.Tensor:
    n_rows, n_cols = x.shape
    # BLOCK_SIZE must be a power of 2 >= n_cols
    BLOCK_SIZE = triton.next_power_of_2(n_cols)
    output = torch.empty_like(x)

    # One program per row
    grid = (n_rows,)

    softmax_kernel[grid](
        output, x,
        x.stride(0), output.stride(0),
        n_cols,
        BLOCK_SIZE=BLOCK_SIZE,
    )
    return output


# Test
x = torch.randn(128, 512, device='cuda')
triton_out = softmax(x)
torch_out = torch.softmax(x, dim=1)

print(f"Max diff: {(triton_out - torch_out).abs().max():.2e}")
print(f"Rows sum to 1? {triton_out.sum(dim=1)[:5]}")
print("Correct!" if torch.allclose(triton_out, torch_out, atol=1e-5) else "MISMATCH!")

Max diff: 5.59e-09
Rows sum to 1? tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000], device='cuda:0')
Correct!


### What just happened?

- **No intermediate tensors** — max, exp, sum all happen in fast on-chip SRAM
- **One global memory read** (input) and **one write** (output) vs 5+ for unfused PyTorch
- The `other=-float('inf')` in `tl.load` fills masked positions with -inf, so they don't affect max/softmax

---
## 3. Matrix Multiplication — The Classic GPU Workload

This is where Triton really shows its power. We'll implement tiled matmul that rivals cuBLAS.

### The Tiling Strategy

```
        N cols
    ┌─────────────┐
    │             │ K
B = │      B      │ rows
    │             │
    └─────────────┘

         K cols          N cols
    ┌───────────┐   ┌──────────────┐
    │           │   │  ┌────┐      │
M   │     A     │ × │  │tile│      │ M
rows│           │   │  │C   │      │ rows
    │           │   │  └────┘      │
    └───────────┘   └──────────────┘

Each program computes one BLOCK_M×BLOCK_N tile of C
by accumulating partial results over K dimension.
```

In [7]:
@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    # 2D grid: each program computes a BLOCK_M x BLOCK_N tile of output
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    # Offsets for the output tile
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)  # [BLOCK_M]
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)  # [BLOCK_N]
    offs_k = tl.arange(0, BLOCK_K)                     # [BLOCK_K]

    # Pointers to the first tiles of A and B
    # A tile: rows=offs_m, cols=offs_k  →  shape [BLOCK_M, BLOCK_K]
    # B tile: rows=offs_k, cols=offs_n  →  shape [BLOCK_K, BLOCK_N]
    a_ptrs = a_ptr + offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak
    b_ptrs = b_ptr + offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn

    # Accumulate in fp32 for numerical stability
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    # Loop over K dimension in chunks of BLOCK_K
    for k_start in range(0, K, BLOCK_K):
        # Masking for boundary tiles
        a_mask = (offs_m[:, None] < M) & ((k_start + offs_k[None, :]) < K)
        b_mask = ((k_start + offs_k[:, None]) < K) & (offs_n[None, :] < N)

        a = tl.load(a_ptrs, mask=a_mask, other=0.0)
        b = tl.load(b_ptrs, mask=b_mask, other=0.0)

        acc += tl.dot(a, b)  # BLOCK_M×BLOCK_K @ BLOCK_K×BLOCK_N

        # Advance pointers to next K-chunk
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    # Store the result tile
    c_ptrs = c_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn
    c_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    tl.store(c_ptrs, acc, mask=c_mask)

In [8]:
def matmul(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    assert a.shape[1] == b.shape[0], "Incompatible dimensions"
    M, K = a.shape
    K, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=torch.float32)

    BLOCK_M, BLOCK_N, BLOCK_K = 64, 64, 32

    grid = (
        triton.cdiv(M, BLOCK_M),
        triton.cdiv(N, BLOCK_N),
    )

    matmul_kernel[grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
    )
    return c


# Test
M, N, K = 512, 256, 128
a = torch.randn(M, K, device='cuda', dtype=torch.float32)
b = torch.randn(K, N, device='cuda', dtype=torch.float32)

triton_out = matmul(a, b)
torch_out = a @ b

print(f"Max diff: {(triton_out - torch_out).abs().max():.2e}")
print("Correct!" if torch.allclose(triton_out, torch_out, atol=1e-1) else "MISMATCH!")

Max diff: 4.35e-02
MISMATCH!


---
## 4. Autotuning — Let Triton Find the Best Configuration

Different GPUs and problem sizes benefit from different block sizes.
Triton's `@triton.autotune` automatically benchmarks multiple configurations and picks the fastest.

In [ ]:
@triton.autotune(
    configs=[
        triton.Config({'BLOCK_SIZE': 64},  num_warps=2),
        triton.Config({'BLOCK_SIZE': 128}, num_warps=4),
        triton.Config({'BLOCK_SIZE': 256}, num_warps=4),
        triton.Config({'BLOCK_SIZE': 512}, num_warps=8),
        triton.Config({'BLOCK_SIZE': 1024}, num_warps=8),
    ],
    key=['n_elements'],  # re-tune when this value changes
)
@triton.jit
def relu_kernel(
    x_ptr, output_ptr, n_elements,
    BLOCK_SIZE: tl.constexpr,
):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask)
    output = tl.maximum(x, 0.0)
    tl.store(output_ptr + offsets, output, mask=mask)


def relu(x: torch.Tensor) -> torch.Tensor:
    output = torch.empty_like(x)
    n = x.numel()
    grid = lambda meta: (triton.cdiv(n, meta['BLOCK_SIZE']),)
    relu_kernel[grid](x, output, n)
    return output


x = torch.randn(1_000_000, device='cuda')
triton_out = relu(x)
torch_out = torch.relu(x)
print(f"Max diff: {(triton_out - torch_out).abs().max():.2e}")
print("Correct!" if torch.allclose(triton_out, torch_out) else "MISMATCH!")

---
## 5. Benchmarking with `triton.testing`

Triton includes a built-in benchmarking utility that generates comparison plots.

In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=['size'],
        x_vals=[2**i for i in range(12, 25)],  # 4K to 16M elements
        line_arg='provider',
        line_vals=['triton', 'torch'],
        line_names=['Triton', 'PyTorch'],
        styles=[('blue', '-'), ('green', '-')],
        ylabel='GB/s',
        plot_name='vector-add-performance',
        args={},
    )
)
def benchmark(size, provider):
    x = torch.rand(size, device='cuda', dtype=torch.float32)
    y = torch.rand(size, device='cuda', dtype=torch.float32)
    quantiles = [0.5, 0.2, 0.8]  # median, 20th, 80th percentile

    if provider == 'torch':
        ms, min_ms, max_ms = triton.testing.do_bench(lambda: x + y, quantiles=quantiles)
    elif provider == 'triton':
        ms, min_ms, max_ms = triton.testing.do_bench(lambda: add(x, y), quantiles=quantiles)

    gbps = lambda ms: 3 * x.numel() * x.element_size() * 1e-9 / (ms * 1e-3)
    return gbps(ms), gbps(max_ms), gbps(min_ms)


benchmark.run(print_data=True)

---
## 6. Practical Pattern: Fused LayerNorm

LayerNorm is one of the most common operations in Transformers.
It's memory-bound and benefits enormously from fusion.

In [ ]:
@triton.jit
def layernorm_kernel(
    x_ptr, output_ptr,
    weight_ptr, bias_ptr,
    stride,
    N,
    eps: tl.constexpr,
    BLOCK_SIZE: tl.constexpr,
):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < N

    # Load one row
    x = tl.load(x_ptr + row * stride + cols, mask=mask, other=0.0).to(tl.float32)

    # Compute mean and variance
    mean = tl.sum(x, axis=0) / N
    x_centered = x - mean
    var = tl.sum(x_centered * x_centered, axis=0) / N
    rstd = 1.0 / tl.sqrt(var + eps)

    # Normalize and apply affine transform
    x_hat = x_centered * rstd
    w = tl.load(weight_ptr + cols, mask=mask)
    b = tl.load(bias_ptr + cols, mask=mask)
    output = x_hat * w + b

    tl.store(output_ptr + row * stride + cols, output, mask=mask)


def layernorm(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor, eps=1e-5):
    M, N = x.shape
    output = torch.empty_like(x)
    BLOCK_SIZE = triton.next_power_of_2(N)

    layernorm_kernel[(M,)](
        x, output, weight, bias,
        x.stride(0), N, eps,
        BLOCK_SIZE=BLOCK_SIZE,
    )
    return output


# Test
M, N = 256, 768  # typical transformer hidden dim
x = torch.randn(M, N, device='cuda')
w = torch.ones(N, device='cuda')
b = torch.zeros(N, device='cuda')

triton_out = layernorm(x, w, b)
torch_out = torch.nn.functional.layer_norm(x, [N], w, b)

print(f"Max diff: {(triton_out - torch_out).abs().max():.2e}")
print("Correct!" if torch.allclose(triton_out, torch_out, atol=1e-5) else "MISMATCH!")

---
## 7. Essential Triton Language (`tl`) Reference

### Memory Operations
| Function | Description |
|---|---|
| `tl.load(ptr, mask, other)` | Load from global memory. `other` fills masked positions |
| `tl.store(ptr, val, mask)` | Store to global memory |
| `tl.atomic_add(ptr, val)` | Atomic addition (for reductions) |

### Creation
| Function | Description |
|---|---|
| `tl.arange(start, end)` | 1D range (like `range()`) |
| `tl.zeros(shape, dtype)` | Zero-filled tensor |
| `tl.full(shape, val, dtype)` | Constant-filled tensor |

### Math
| Function | Description |
|---|---|
| `tl.exp(x)` | Exponential |
| `tl.log(x)` | Natural log |
| `tl.sqrt(x)` | Square root |
| `tl.abs(x)` | Absolute value |
| `tl.maximum(x, y)` | Element-wise max |
| `tl.dot(a, b)` | Matrix multiply (for tiles) |
| `tl.where(cond, x, y)` | Conditional select |

### Reductions
| Function | Description |
|---|---|
| `tl.sum(x, axis)` | Sum reduction |
| `tl.max(x, axis)` | Max reduction |
| `tl.min(x, axis)` | Min reduction |

### Indexing & Control
| Function | Description |
|---|---|
| `tl.program_id(axis)` | Current program's ID on given axis |
| `tl.num_programs(axis)` | Total programs on given axis |
| `x[:, None]` | Broadcast (add dimension) |
| `tl.constexpr` | Marks a compile-time constant |

---
## 8. Common Patterns & Tips

### Pattern 1: Pointer Arithmetic for 2D Tensors
```python
# To index element [i, j] in a 2D tensor:
ptr + i * stride_row + j * stride_col

# For a tile of elements:
row_offsets = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)  # shape: [BLOCK_M]
col_offsets = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)  # shape: [BLOCK_N]
ptrs = ptr + row_offsets[:, None] * stride_row + col_offsets[None, :] * stride_col
# ptrs shape: [BLOCK_M, BLOCK_N]
```

### Pattern 2: Online Reductions (Loop over K)
```python
acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
for k in range(0, K, BLOCK_K):
    a = tl.load(a_ptrs, mask=...)
    b = tl.load(b_ptrs, mask=...)
    acc += tl.dot(a, b)
    a_ptrs += BLOCK_K * stride  # advance pointers
```

### Pattern 3: Atomic Operations for Parallel Reductions
```python
# When multiple programs write to the same output location
tl.atomic_add(output_ptr + offsets, values, mask=mask)
```

### Tips

1. **BLOCK_SIZE should be a power of 2** — hardware is optimized for this
2. **Use `tl.constexpr` for all sizes** — enables compile-time optimization
3. **Accumulate in fp32** — even if inputs are fp16, accumulate in fp32 for stability
4. **Minimize global memory access** — fuse operations to keep data in SRAM
5. **Use `num_warps`** — more warps help hide memory latency (default is 4)
6. **Use `num_stages`** — enables software pipelining of memory loads (default is 2)

---
## 9. Debugging Triton Kernels

Triton kernels can't use `print()`. Here are debugging strategies:

### Strategy 1: Test with tiny inputs
```python
x = torch.tensor([1.0, 2.0, 3.0, 4.0], device='cuda')
# Use BLOCK_SIZE=4 so one program handles everything
```

### Strategy 2: Compare against PyTorch reference
```python
triton_out = my_kernel_wrapper(x)
torch_out = torch_reference(x)
print(torch.allclose(triton_out, torch_out, atol=1e-5))
print((triton_out - torch_out).abs().max())  # max element-wise error
```

### Strategy 3: Use `tl.device_print` (Triton >= 2.1)
```python
@triton.jit
def debug_kernel(x_ptr, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    x = tl.load(x_ptr + offs)
    tl.device_print("pid", pid)
    tl.device_print("x", x)
```

### Strategy 4: Write intermediate values to global memory
```python
@triton.jit
def debug_kernel(x_ptr, debug_ptr, ...):
    # ... compute intermediate ...
    tl.store(debug_ptr + offsets, intermediate_value, mask=mask)
```

---
## 10. Exercises

Try implementing these kernels to practice:

1. **GELU activation**: `0.5 * x * (1 + tanh(sqrt(2/pi) * (x + 0.044715 * x^3)))`
2. **Fused bias + ReLU**: Load input, add bias vector, apply ReLU, store
3. **Dropout**: Use `tl.rand()` to generate random mask, scale by `1/(1-p)`
4. **Row-wise L2 norm**: Compute `||x||_2` for each row of a matrix
5. **Cross-entropy loss**: Fuse log-softmax + nll_loss into one kernel

### Resources

- [Official Triton tutorials](https://triton-lang.org/main/getting-started/tutorials/index.html)
- [Triton GitHub](https://github.com/triton-lang/triton)
- [Liger Kernel](https://github.com/linkedin/Liger-Kernel) — production Triton kernels for LLM training
- [Unsloth](https://github.com/unslothai/unsloth) — optimized Triton kernels for fine-tuning